<a href="https://colab.research.google.com/github/potato-yen/114-2-Programming-Language/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -q google-generativeai gspread google-auth google-auth-oauthlib gradio pandas

In [13]:
# -*- coding: utf-8 -*-
from datetime import datetime
import json
import re

import pandas as pd
import gradio as gr
import gspread
import google.generativeai as legacy_genai  # 保留相容性，不直接使用
from google.colab import auth, userdata
from google.auth import default
from google import genai

In [14]:
# Gemini 設定
api_key = userdata.get("gemini")
client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-2.5-flash"


In [15]:
# Google Sheet 設定
SHEET_URL = "https://docs.google.com/spreadsheets/d/1Etj3aomNgs1hroV3JwgRMNMVQ2_BIyKwChgtPRnYxic/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = [
    "日期",
    "類型",
    "科目",
    "分數",
    "整體表現",
    "待加強科目",
    "學習建議"
]

_auth_done = False
_gc = None
_ws = None


In [16]:
def authenticate_google_sheet():
    global _auth_done, _gc

    if not _auth_done:
        auth.authenticate_user()
        creds, _ = default()
        _gc = gspread.authorize(creds)
        _auth_done = True

    return _gc


def get_worksheet():
    global _ws

    gc = authenticate_google_sheet()
    if _ws is None:
        sh = gc.open_by_url(SHEET_URL)
        _ws = sh.worksheet(WORKSHEET_NAME)

    return _ws


def ensure_sheet_columns(ws):
    values = ws.get_all_values()

    if not values:
        ws.append_row(REQUIRED_COLUMNS)
        print("已建立表頭。")
        return

    header = values[0]
    if header[:len(REQUIRED_COLUMNS)] != REQUIRED_COLUMNS:
        raise ValueError(
            "Google Sheet 的表頭格式與程式預期不一致。\n"
            f"目前表頭：{header}\n"
            f"預期表頭：{REQUIRED_COLUMNS}"
        )


In [17]:
def get_user_grades():
    """
    透過終端機輸入成績，直到使用者輸入 q 結束。
    每筆資料格式：
    [日期, 類型, 科目, 分數, 整體表現, 待加強科目, 學習建議]
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []

    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：").strip()
        if subject.lower() == "q":
            break

        if not subject:
            print("科目名稱不能空白，請重新輸入。")
            continue

        grade_raw = input(f"請輸入 {subject} 的成績（0~100）：").strip()

        try:
            grade = int(grade_raw)
        except ValueError:
            print("成績必須是整數，請重新輸入。")
            continue

        if not 0 <= grade <= 100:
            print("成績必須介於 0 到 100 之間，請重新輸入。")
            continue

        today = datetime.now().strftime("%Y-%m-%d")
        grades.append([today, "成績紀錄", subject, grade, "", "", ""])
        print(f"已記錄：日期={today}，科目={subject}，分數={grade}\n")

    return grades


In [18]:
def analyze_grades(grades):
    """
    根據輸入成績做基本統計，避免讓 AI 自己亂算。
    """
    if not grades:
        return None

    subject_scores = [{"subject": row[2], "score": row[3]} for row in grades]
    scores = [item["score"] for item in subject_scores]

    average_score = round(sum(scores) / len(scores), 1)
    highest_item = max(subject_scores, key=lambda x: x["score"])
    lowest_item = min(subject_scores, key=lambda x: x["score"])
    weak_subjects = [item["subject"] for item in subject_scores if item["score"] < 60]

    return {
        "count": len(scores),
        "average": average_score,
        "highest_subject": highest_item["subject"],
        "highest_score": highest_item["score"],
        "lowest_subject": lowest_item["subject"],
        "lowest_score": lowest_item["score"],
        "weak_subjects": weak_subjects
    }


def extract_json_object(text):
    """
    從模型輸出中抓取 JSON 物件。
    """
    if not text:
        raise ValueError("模型沒有回傳內容。")

    text = text.strip()

    # 先嘗試整段直接解析
    try:
        return json.loads(text)
    except Exception:
        pass

    # 再抓 ```json ... ``` 或一般大括號區塊
    match = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if not match:
        match = re.search(r"(\{.*\})", text, re.DOTALL)

    if not match:
        raise ValueError(f"找不到有效 JSON：{text}")

    return json.loads(match.group(1))


def get_ai_summary_structured(grades, stats):
    """
    呼叫 Gemini，要求輸出固定 JSON，避免內容雜亂。
    """
    grade_lines = []
    for row in grades:
        grade_lines.append(f"{row[2]}：{row[3]}分")

    prompt_text = f'''
你是一個學習成績整理助手。請根據以下成績資料，輸出「簡短、規整、可直接寫入 Google Sheet」的分析結果。

【成績資料】
{chr(10).join(grade_lines)}

【已計算統計】
- 科目數：{stats["count"]}
- 平均分：{stats["average"]}
- 最高分：{stats["highest_subject"]} {stats["highest_score"]}分
- 最低分：{stats["lowest_subject"]} {stats["lowest_score"]}分
- 低於60分科目：{", ".join(stats["weak_subjects"]) if stats["weak_subjects"] else "無"}

【輸出限制】
1. 只能輸出 JSON。
2. 不要加 markdown，不要加說明，不要加反引號。
3. JSON 格式固定如下：
{{
  "overall": "整體表現，25字內",
  "weak_subjects": "待加強科目整理，20字內；若無則寫無",
  "advice": "學習建議，40字內"
}}
4. 內容要精簡、自然、具體，不要空話。
5. 不要捏造不存在的資料，不要提到排名、標準差、班級比較。
'''.strip()

    print("\n--- 正在呼叫 AI 模型生成結構化摘要... ---")
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt_text
        )

        raw_text = response.text
        data = extract_json_object(raw_text)

        return {
            "overall": str(data.get("overall", "")).strip()[:25],
            "weak_subjects": str(data.get("weak_subjects", "")).strip()[:20],
            "advice": str(data.get("advice", "")).strip()[:40],
            "raw_text": raw_text
        }

    except Exception as e:
        print(f"呼叫 AI 或解析輸出時發生錯誤：{e}")
        return {
            "overall": "AI 摘要生成失敗",
            "weak_subjects": "無法分析",
            "advice": "請稍後再試",
            "raw_text": ""
        }


In [19]:
def build_summary_row(ai_result):
    today = datetime.now().strftime("%Y-%m-%d")
    return [
        today,
        "AI摘要",
        "",
        "",
        ai_result["overall"],
        ai_result["weak_subjects"],
        ai_result["advice"]
    ]


def append_records_to_sheet(grades, ai_result):
    ws = get_worksheet()
    ensure_sheet_columns(ws)
    ws.append_rows(grades)
    ws.append_row(build_summary_row(ai_result))
    return ws


def grades_to_dataframe(grades):
    if not grades:
        return pd.DataFrame(columns=["科目", "分數"])
    return pd.DataFrame([{"科目": row[2], "分數": row[3]} for row in grades])


def format_stats_html(stats):
    if not stats:
        return "<div class='empty-note'>尚未產生統計資料</div>"

    weak_text = "、".join(stats["weak_subjects"]) if stats["weak_subjects"] else "無"
    return f"""
    <div class='stats-grid'>
      <div class='stat-card sandy'>
        <div class='stat-label'>平均分</div>
        <div class='stat-value'>{stats['average']}</div>
      </div>
      <div class='stat-card mist'>
        <div class='stat-label'>最高分</div>
        <div class='stat-value'>{stats['highest_subject']} {stats['highest_score']}</div>
      </div>
      <div class='stat-card sage'>
        <div class='stat-label'>最低分</div>
        <div class='stat-value'>{stats['lowest_subject']} {stats['lowest_score']}</div>
      </div>
      <div class='stat-card lilac'>
        <div class='stat-label'>待加強</div>
        <div class='stat-value'>{weak_text}</div>
      </div>
    </div>
    """


def card_html(title, body, tone="sandy"):
    body = body if body else "尚未生成內容"
    return f"""
    <div class='analysis-card {tone}'>
      <div class='analysis-title'>{title}</div>
      <div class='analysis-body'>{body}</div>
    </div>
    """


def add_grade_entry(subject, score, current_grades):
    current_grades = current_grades or []
    subject = (subject or "").strip()

    default_stats = format_stats_html(None)
    default_overall = card_html("整體表現", "")
    default_weak = card_html("待加強科目", "", "mist")
    default_advice = card_html("學習建議", "", "sage")

    if not subject:
        return current_grades, grades_to_dataframe(current_grades), "", None, "<div class='status error'>科目名稱不能空白。</div>", default_stats, default_overall, default_weak, default_advice, None

    try:
        score = int(score)
    except Exception:
        return current_grades, grades_to_dataframe(current_grades), "", None, "<div class='status error'>分數必須是整數。</div>", default_stats, default_overall, default_weak, default_advice, None

    if not 0 <= score <= 100:
        return current_grades, grades_to_dataframe(current_grades), "", None, "<div class='status error'>分數必須介於 0 到 100 之間。</div>", default_stats, default_overall, default_weak, default_advice, None

    today = datetime.now().strftime("%Y-%m-%d")
    current_grades.append([today, "成績紀錄", subject, score, "", "", ""])
    return current_grades, grades_to_dataframe(current_grades), "", None, f"<div class='status ok'>已加入：{subject} {score} 分</div>", default_stats, default_overall, default_weak, default_advice, None


def generate_analysis_preview(current_grades):
    current_grades = current_grades or []
    if not current_grades:
        return format_stats_html(None), card_html("整體表現", ""), card_html("待加強科目", "", "mist"), card_html("學習建議", "", "sage"), "<div class='status error'>請先加入至少一筆成績。</div>", None

    stats = analyze_grades(current_grades)
    ai_result = get_ai_summary_structured(current_grades, stats)
    return format_stats_html(stats), card_html("整體表現", ai_result["overall"]), card_html("待加強科目", ai_result["weak_subjects"], "mist"), card_html("學習建議", ai_result["advice"], "sage"), "<div class='status ok'>分析完成，可以寫入 Google Sheet。</div>", ai_result


def save_current_batch(current_grades, ai_result):
    current_grades = current_grades or []
    if not current_grades:
        return current_grades, grades_to_dataframe(current_grades), format_stats_html(None), card_html("整體表現", ""), card_html("待加強科目", "", "mist"), card_html("學習建議", "", "sage"), "<div class='status error'>沒有可寫入的成績資料。</div>", None

    if not ai_result:
        return current_grades, grades_to_dataframe(current_grades), format_stats_html(None), card_html("整體表現", ""), card_html("待加強科目", "", "mist"), card_html("學習建議", "", "sage"), "<div class='status error'>請先按「生成分析」再寫入。</div>", ai_result

    try:
        append_records_to_sheet(current_grades, ai_result)
        return [], grades_to_dataframe([]), format_stats_html(None), card_html("整體表現", ""), card_html("待加強科目", "", "mist"), card_html("學習建議", "", "sage"), f"<div class='status ok'>已成功寫入 {len(current_grades)} 筆成績紀錄與 1 筆 AI 摘要。</div>", None
    except gspread.exceptions.APIError as e:
        return current_grades, grades_to_dataframe(current_grades), format_stats_html(None), card_html("整體表現", ai_result["overall"]), card_html("待加強科目", ai_result["weak_subjects"], "mist"), card_html("學習建議", ai_result["advice"], "sage"), f"<div class='status error'>Google Sheets API 錯誤：{e.response.text}</div>", ai_result
    except Exception as e:
        return current_grades, grades_to_dataframe(current_grades), format_stats_html(None), card_html("整體表現", ai_result["overall"]), card_html("待加強科目", ai_result["weak_subjects"], "mist"), card_html("學習建議", ai_result["advice"], "sage"), f"<div class='status error'>發生錯誤：{e}</div>", ai_result


def reset_current_batch():
    return [], grades_to_dataframe([]), "", None, format_stats_html(None), card_html("整體表現", ""), card_html("待加強科目", "", "mist"), card_html("學習建議", "", "sage"), "<div class='status'>已清空本次輸入。</div>", None

In [24]:
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Noto+Sans+TC:wght@400;500;700&family=Cormorant+Garamond:wght@600;700&display=swap');

:root {
  --paper: #f5f3ee;
  --paper-soft: #f8f5ef;
  --paper-deep: #e6dfd2;
  --paper-panel: #efe8dc;

  --ink: #171717;
  --ink-soft: #5e574f;

  --line: #d5cec1;
  --line-soft: #c9bfae;

  --sand: #e8dfcf;
  --mist: #dde7f1;
  --sage: #dbe6df;
  --lilac: #e3deeb;

  --button: #111111;

  --panel-dark: #4a4953;
  --panel-dark-2: #43424b;
  --panel-text-light: #f3efe7;
  --panel-text-soft: #d7d0c5;
}

/* ===== Base ===== */
html, body, .gradio-container {
  background: var(--paper) !important;
  font-family: 'Noto Sans TC', sans-serif !important;
  color: var(--ink) !important;
}

body {
  color: var(--ink) !important;
}

#app-shell {
  max-width: 1180px;
  margin: 0 auto;
}

/* ===== Hero ===== */
.hero-wrap {
  padding: 24px 8px 10px 8px;
}

.eyebrow {
  font-size: 15px;
  letter-spacing: 0.02em;
  color: var(--ink-soft);
  margin-bottom: 10px;
}

.hero-title {
  font-family: 'Cormorant Garamond', serif;
  font-size: 68px;
  line-height: 0.95;
  margin: 0;
  color: var(--ink);
}

.hero-subtitle {
  font-size: 20px;
  line-height: 1.8;
  color: var(--ink-soft);
  max-width: 700px;
  margin-top: 18px;
}

/* ===== Outer Cards ===== */
.input-card, .preview-card {
  border: 1px solid var(--line) !important;
  border-radius: 28px !important;
  box-shadow: none !important;
  padding: 12px !important;
}

.input-card {
  background: var(--paper-soft) !important;
}

.preview-card {
  background: var(--paper-deep) !important;
}

/* 只洗掉一般容器，不碰 dark-panel */
.input-card > div:not(.dark-panel),
.preview-card > div:not(.dark-panel),
.input-card .block:not(.dark-panel),
.preview-card .block:not(.dark-panel),
.input-card .gr-box:not(.dark-panel),
.preview-card .gr-box:not(.dark-panel),
.input-card .gr-panel:not(.dark-panel),
.preview-card .gr-panel:not(.dark-panel),
.input-card .gr-form:not(.dark-panel),
.preview-card .gr-form:not(.dark-panel) {
  background: transparent !important;
  color: var(--ink) !important;
  border-color: transparent !important;
  box-shadow: none !important;
}

/* ===== Titles ===== */
.section-title h3 {
  font-family: 'Cormorant Garamond', serif;
  font-size: 34px;
  margin: 0 0 6px 0;
  color: var(--ink);
}

.section-note {
  color: var(--ink-soft);
  font-size: 15px;
  margin-bottom: 14px;
}

/* ===== General Labels ===== */
label, .gr-form label, .gr-block label, .gradio-container label {
  color: var(--ink) !important;
  font-weight: 600 !important;
}

/* ===== Inputs ===== */
textarea,
input,
select,
.gr-textbox textarea,
.gr-textbox input,
.gr-number input {
  background: var(--paper-soft) !important;
  color: var(--ink) !important;
  border: 1px solid var(--line-soft) !important;
  border-radius: 14px !important;
  box-shadow: none !important;
}

textarea::placeholder,
input::placeholder {
  color: #8a8177 !important;
}

/* ===== Buttons ===== */
button.primary-btn {
  background: var(--button) !important;
  color: #fff !important;
  border-radius: 999px !important;
  border: none !important;
}

button.secondary-btn {
  background: transparent !important;
  color: var(--ink) !important;
  border-radius: 999px !important;
  border: 1px solid var(--ink-soft) !important;
}

button {
  box-shadow: none !important;
}

/* ===== Status ===== */
.status {
  border: 1px solid var(--line);
  background: rgba(255,255,255,0.55);
  border-radius: 18px;
  padding: 12px 16px;
  color: var(--ink-soft);
  font-size: 15px;
}

.status.ok {
  background: rgba(219, 230, 223, 0.72);
  color: var(--ink);
}

.status.error {
  background: rgba(226, 221, 236, 0.72);
  color: var(--ink);
}

/* ===== Stats / Analysis ===== */
.stats-grid {
  display: grid;
  grid-template-columns: repeat(2, minmax(0, 1fr));
  gap: 12px;
}

.stat-card {
  border-radius: 22px;
  padding: 18px;
  min-height: 110px;
  border: 1px solid rgba(23,23,23,0.08);
}

.analysis-card {
  border-radius: 24px;
  padding: 20px;
  min-height: 148px;
  border: 1px solid rgba(23,23,23,0.08);
}

.sandy { background: var(--sand); }
.mist  { background: var(--mist); }
.sage  { background: var(--sage); }
.lilac { background: var(--lilac); }

.stat-label, .analysis-title {
  font-size: 14px;
  color: var(--ink-soft);
  margin-bottom: 10px;
}

.stat-value {
  font-family: 'Cormorant Garamond', serif;
  font-size: 34px;
  line-height: 1.15;
  color: var(--ink);
}

.analysis-title {
  font-family: 'Cormorant Garamond', serif;
  font-size: 30px;
  color: var(--ink);
}

.analysis-body {
  font-size: 17px;
  line-height: 1.9;
  color: var(--ink);
}

.empty-note {
  color: var(--ink-soft);
  padding: 20px 2px;
  font-size: 15px;
}

/* ===== Markdown / HTML 文字 ===== */
.gradio-container .prose,
.gradio-container .prose p,
.gradio-container .prose span,
.gradio-container .prose div,
.gradio-container .prose li,
.gradio-container .markdown,
.gradio-container .markdown p,
.gradio-container .markdown span,
.gradio-container .html-container,
.gradio-container .html-container p,
.gradio-container .html-container span,
.gradio-container .html-container div {
  color: var(--ink) !important;
}

/* ===== Default Tables ===== */
table,
thead,
tbody,
tr,
th,
td {
  background: transparent !important;
  color: var(--ink) !important;
  border-color: var(--line-soft) !important;
}

th {
  background: rgba(255,255,255,0.35) !important;
  font-weight: 700 !important;
}

td {
  background: rgba(255,255,255,0.18) !important;
}

/* ===== Dark Panel ===== */
/* 給左側上下兩塊深色區塊用 */
.dark-panel {
  background: var(--panel-dark) !important;
  border-radius: 18px !important;
  padding: 14px 14px 12px 14px !important;
  border: 1px solid rgba(255,255,255,0.06) !important;
  box-shadow: none !important;
}

/* 防止 dark-panel 內層又被透明化或洗色 */
.dark-panel > div,
.dark-panel .block,
.dark-panel .gr-box,
.dark-panel .gr-panel,
.dark-panel .gr-form,
.dark-panel [data-testid="block-root"] {
  background: transparent !important;
  box-shadow: none !important;
  border-color: transparent !important;
}

.dark-panel .panel-label {
  color: var(--panel-text-light) !important;
  font-size: 15px;
  font-weight: 700;
  margin-bottom: 12px;
}

.dark-panel .panel-note {
  color: var(--panel-text-soft) !important;
  font-size: 13px;
  margin-bottom: 10px;
}

/* dark-panel 內所有一般文字 */
.dark-panel,
.dark-panel p,
.dark-panel span,
.dark-panel div,
.dark-panel small,
.dark-panel label {
  color: var(--panel-text-light) !important;
}

/* dark-panel 內的欄位標籤 */
.dark-panel label,
.dark-panel .gr-form label,
.dark-panel .gr-block label {
  color: var(--panel-text-light) !important;
  font-weight: 600 !important;
}

/* dark-panel 裡的輸入框，維持淺底 */
.dark-panel textarea,
.dark-panel input,
.dark-panel select,
.dark-panel .gr-textbox textarea,
.dark-panel .gr-textbox input,
.dark-panel .gr-number input {
  background: #f1ece4 !important;
  color: var(--ink) !important;
  border: 1px solid rgba(255,255,255,0.16) !important;
  border-radius: 14px !important;
}

/* dark-panel 裡的 placeholder */
.dark-panel textarea::placeholder,
.dark-panel input::placeholder {
  color: #8e8579 !important;
}

/* dark-panel 裡的表格 */
.dark-panel table,
.dark-panel thead,
.dark-panel tbody,
.dark-panel tr,
.dark-panel th,
.dark-panel td {
  background: transparent !important;
  color: var(--panel-text-light) !important;
  border-color: rgba(255,255,255,0.16) !important;
}

.dark-panel th {
  background: rgba(255,255,255,0.08) !important;
  color: var(--panel-text-light) !important;
}

.dark-panel td {
  background: rgba(255,255,255,0.02) !important;
  color: var(--panel-text-light) !important;
}

/* dark-panel 內 dataframe 常見包裝層 */
.dark-panel .table-wrap,
.dark-panel .dataframe,
.dark-panel .dataframe-container {
  background: transparent !important;
  color: var(--panel-text-light) !important;
}

/* ===== Optional: records 區可額外指定 class 時更穩 ===== */
.records-table table,
.records-table thead,
.records-table tbody,
.records-table tr,
.records-table th,
.records-table td {
  color: var(--panel-text-light) !important;
  border-color: rgba(255,255,255,0.16) !important;
}

/* ===== Footer ===== */
footer {
  display: none !important;
}
"""

In [ ]:
with gr.Blocks(css=CUSTOM_CSS) as demo:
    grades_state = gr.State([])
    ai_state = gr.State(None)

    with gr.Column(elem_id="app-shell"):
        gr.HTML("""
        <div class='hero-wrap'>
          <div class='eyebrow'>Grade Journal · 成績一本通</div>
          <h1 class='hero-title'>整理成績，生成摘要，<br>再安靜地寫進 Google Sheet。</h1>
          <div class='hero-subtitle'>
            這個介面延續第一部分的資料流程：先輸入成績，再由程式統計與 Gemini 摘要共同生成分析，最後追加寫入工作表二，不覆蓋舊資料。
          </div>
        </div>
        """)

        with gr.Row(equal_height=True):
            with gr.Column(scale=5):
                with gr.Group(elem_classes=["input-card"]):
                    gr.HTML("""
                    <div class='section-title'><h3>本次輸入</h3></div>
                    <div class='section-note'>逐筆加入科目與分數，建立這一批待分析資料。</div>
                    """)

                    with gr.Group(elem_classes=["dark-panel"]):
                        with gr.Row():
                            subject_input = gr.Textbox(
                                label="科目名稱",
                                placeholder="例如：國文、數學、英文"
                            )
                            score_input = gr.Number(
                                label="分數",
                                precision=0,
                                minimum=0,
                                maximum=100,
                                placeholder="0 ~ 100"
                            )

                    with gr.Row():
                        add_btn = gr.Button("加入成績", elem_classes=["primary-btn"])
                        reset_btn = gr.Button("清空本次資料", elem_classes=["secondary-btn"])

                    with gr.Group(elem_classes=["dark-panel"]):
                        gr.HTML("<div class='panel-label'>本次已加入資料</div>")
                        grade_table = gr.Dataframe(
                            headers=["科目", "分數"],
                            datatype=["str", "number"],
                            value=grades_to_dataframe([]),
                            interactive=False,
                            row_count=(0, "dynamic"),
                            col_count=(2, "fixed"),
                            show_label=False,
                            elem_classes=["records-table"]
                        )

            with gr.Column(scale=7):
                with gr.Group(elem_classes=["preview-card"]):
                    gr.HTML("""
                    <div class='section-title'><h3>分析預覽</h3></div>
                    <div class='section-note'>先看統計與 AI 摘要，再決定是否寫入 Google Sheet。</div>
                    """)
                    stats_html = gr.HTML(format_stats_html(None))
                    with gr.Row():
                        overall_html = gr.HTML(card_html("整體表現", ""))
                        weak_html = gr.HTML(card_html("待加強科目", "", "mist"))
                        advice_html = gr.HTML(card_html("學習建議", "", "sage"))

        with gr.Row():
            analyze_btn = gr.Button("生成分析", elem_classes=["primary-btn"])
            save_btn = gr.Button("寫入 Google Sheet", elem_classes=["secondary-btn"])

        status_html = gr.HTML("<div class='status'>請先加入成績，再生成分析。</div>")

    add_btn.click(
        fn=add_grade_entry,
        inputs=[subject_input, score_input, grades_state],
        outputs=[grades_state, grade_table, subject_input, score_input, status_html, stats_html, overall_html, weak_html, advice_html, ai_state]
    )

    analyze_btn.click(
        fn=generate_analysis_preview,
        inputs=[grades_state],
        outputs=[stats_html, overall_html, weak_html, advice_html, status_html, ai_state]
    )

    save_btn.click(
        fn=save_current_batch,
        inputs=[grades_state, ai_state],
        outputs=[grades_state, grade_table, stats_html, overall_html, weak_html, advice_html, status_html, ai_state]
    )

    reset_btn.click(
        fn=reset_current_batch,
        inputs=[],
        outputs=[grades_state, grade_table, subject_input, score_input, stats_html, overall_html, weak_html, advice_html, status_html, ai_state]
    )

demo.launch(debug=True)

/tmp/ipykernel_3847/480679984.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/04/01 14:18:34 [W] [service.go:132] login to server failed: tls: failed to verify certificate: x509: certificate has expired or is not yet valid: current time 2026-04-01T14:18:34Z is after 2026-04-01T07:01:35Z


<IPython.core.display.Javascript object>


--- 正在呼叫 AI 模型生成結構化摘要... ---
